# RSNA Knee — Notebook 01: Baseline / Supervision / Fold Audit

This notebook is the **first preparation stage** for continual fine-tuning of the existing ~0.89 DINOv2 ensemble.

It intentionally **does not fine-tune yet**. The uploaded inference notebook has a 20-member manifest checkpoint package, but it does not contain the original training loop or a verified `StudyInstanceUID -> fold` mapping. Re-creating folds arbitrarily and calling them the original folds could cause leakage.

### What this notebook does

1. Finds the RSNA Knee training data and the existing DINOv2 weight package.
2. Audits `train.csv` labels while treating missing labels as **unknown**, not negative.
3. Reads the existing `manifest.json` and builds a checkpoint/member registry.
4. Inspects checkpoint structure without loading all checkpoints into memory at once.
5. Searches for a **real original fold-map artifact**; it never invents one silently.
6. Writes masked-label tables and expert subsets for later fine-tuning.
7. Creates a compact `stage1_bundle.zip` that becomes an input to Notebook 02.

### Important rule

If no original study-to-fold mapping is found, Notebook 02 must **not** pretend the checkpoint fold IDs define validation studies. We can still fine-tune later, but validation must be designed explicitly.

In [ ]:
from __future__ import annotations

import os
import gc
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

WORK = Path("/kaggle/working")
OUT = WORK / "rsna_stage1_bundle"
OUT.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("output:", OUT)

## 1. Locate competition data and the existing 20-member weight package

In [ ]:
def find_train_root() -> Path:
    explicit = os.environ.get("KNEE_INPUT_DIR", "").strip()
    candidates = []
    if explicit:
        candidates.append(Path(explicit))

    candidates += [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("data"),
        Path("."),
    ]

    for c in candidates:
        if (c / "train.csv").is_file() and (c / "train_series.csv").is_file():
            return c

    base = Path("/kaggle/input")
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            root = Path(root)
            if (root / "train.csv").is_file() and (root / "train_series.csv").is_file():
                return root

    raise FileNotFoundError(
        "RSNA Knee training data not found. Attach the competition dataset or set KNEE_INPUT_DIR."
    )


def valid_weight_package(root: Path) -> bool:
    root = Path(root)
    manifest_path = root / "manifest.json"
    if not manifest_path.is_file():
        return False

    try:
        manifest = json.loads(manifest_path.read_text())
    except Exception:
        return False

    members = manifest.get("members")
    if not isinstance(members, list) or len(members) == 0:
        return False

    missing = [
        str(m.get("file"))
        for m in members
        if not (root / str(m.get("file"))).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            f"Weight package exists but is incomplete. First missing checkpoint: {missing[0]}"
        )
    return True


def find_weights() -> Path:
    explicit = os.environ.get("KNEE_WEIGHTS_DIR", "").strip()
    candidates = []
    if explicit:
        candidates.append(Path(explicit))

    candidates += [Path("/kaggle/input/rsna-knee-weights")]

    seen = set()
    for p in candidates:
        key = str(p)
        if key in seen:
            continue
        seen.add(key)
        if valid_weight_package(p):
            return p

    base = Path("/kaggle/input")
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            if "manifest.json" not in files:
                continue
            root = Path(root)
            if valid_weight_package(root):
                return root

    raise FileNotFoundError(
        "Compatible DINO manifest package not found. Attach rsna-knee-weights or set KNEE_WEIGHTS_DIR."
    )


ROOT = find_train_root()
WEIGHTS = find_weights()

print("competition root:", ROOT)
print("weights root:", WEIGHTS)

## 2. Load training metadata and audit label availability

In [ ]:
train_df = pd.read_csv(ROOT / "train.csv", dtype={"StudyInstanceUID": str})
train_series = pd.read_csv(
    ROOT / "train_series.csv",
    dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
)

missing_target_columns = [t for t in TARGETS if t not in train_df.columns]
if missing_target_columns:
    raise KeyError(f"train.csv is missing target columns: {missing_target_columns}")

if "StudyInstanceUID" not in train_df.columns:
    raise KeyError("train.csv must contain StudyInstanceUID")

print("studies:", len(train_df))
print("series:", len(train_series))
print("unique train UIDs:", train_df["StudyInstanceUID"].nunique())
print("duplicate train UIDs:", int(train_df["StudyInstanceUID"].duplicated().sum()))

annotated = train_df[TARGETS].notna().sum()
positive = train_df[TARGETS].sum(skipna=True)
negative = annotated - positive

label_stats = pd.DataFrame({
    "target": TARGETS,
    "annotated": [int(annotated[t]) for t in TARGETS],
    "positive": [float(positive[t]) for t in TARGETS],
    "negative": [float(negative[t]) for t in TARGETS],
})
label_stats["positive_rate_on_annotated"] = (
    label_stats["positive"] / label_stats["annotated"].replace(0, np.nan)
)
label_stats.to_csv(OUT / "label_stats.csv", index=False)

display(label_stats.style.format({"positive_rate_on_annotated": "{:.2%}"}))

n_complete = int(train_df[TARGETS].notna().all(axis=1).sum())
n_any = int(train_df[TARGETS].notna().any(axis=1).sum())
print(f"any image annotation: {n_any} / {len(train_df)}")
print(f"all 12 image annotations: {n_complete} / {len(train_df)}")

### Missing labels remain unknown

For every target we create both:

- the numeric label column;
- a `__mask` column indicating whether that target is actually annotated.

Notebook 02 can therefore use a **masked loss** and avoid treating `NaN` as class `0`.

In [ ]:
masked = train_df[["StudyInstanceUID", *TARGETS]].copy()

for t in TARGETS:
    masked[f"{t}__mask"] = masked[t].notna().astype(np.uint8)

# Keep NaN in the label itself. The mask decides whether a loss term is valid.
masked.to_csv(OUT / "train_labels_masked.csv", index=False)

expert_complete = train_df.loc[
    train_df[TARGETS].notna().all(axis=1),
    ["StudyInstanceUID", *TARGETS]
].copy()
expert_complete.to_csv(OUT / "expert_complete_12.csv", index=False)

partially_labeled = train_df.loc[
    train_df[TARGETS].notna().any(axis=1)
    & ~train_df[TARGETS].notna().all(axis=1),
    ["StudyInstanceUID", *TARGETS]
].copy()
partially_labeled.to_csv(OUT / "partial_or_incomplete_labels.csv", index=False)

print("masked label table:", masked.shape)
print("complete 12-target expert subset:", expert_complete.shape)
print("partial/incomplete subset:", partially_labeled.shape)

## 3. Read the original DINO manifest and build a member registry

In [ ]:
manifest_path = WEIGHTS / "manifest.json"
manifest = json.loads(manifest_path.read_text())
members = manifest["members"]

print("manifest top-level keys:", sorted(manifest.keys()))
print("member count:", len(members))

rows = []
for i, m in enumerate(members):
    cfg = m.get("config") or {}
    ck_path = WEIGHTS / str(m.get("file"))
    rows.append({
        "member_index": i,
        "member_id": m.get("id"),
        "fold": m.get("fold"),
        "holdout": m.get("holdout"),
        "checkpoint_file": m.get("file"),
        "pixel_group": m.get("pixel_group"),
        "variant": cfg.get("variant"),
        "pool": cfg.get("pool", "cls_mean"),
        "prior": cfg.get("prior", False),
        "unfreeze_last": cfg.get("unfreeze_last"),
        "img": cfg.get("img"),
        "group": cfg.get("group"),
        "slices": cfg.get("slices"),
        "crop_mm": cfg.get("crop_mm"),
        "band": json.dumps(cfg.get("band")),
        "slots": json.dumps(cfg.get("slots")),
        "checkpoint_exists": ck_path.is_file(),
        "checkpoint_bytes": ck_path.stat().st_size if ck_path.is_file() else np.nan,
    })

registry = pd.DataFrame(rows)
registry.to_csv(OUT / "member_registry.csv", index=False)
display(registry.sort_values(["fold", "holdout"], ascending=[True, False]))

print("\nMembers per fold:")
display(registry.groupby("fold", dropna=False).size().rename("n_members").to_frame())

print("\nPixel groups:")
display(registry.groupby("pixel_group", dropna=False).size().rename("n_members").to_frame())

## 4. Inspect checkpoint structure safely

In [ ]:
def summarize_checkpoint(path: Path) -> dict:
    ck = torch.load(path, map_location="cpu", weights_only=False)

    info = {
        "file": path.name,
        "container_type": type(ck).__name__,
    }

    if isinstance(ck, dict):
        info["keys"] = json.dumps(sorted(map(str, ck.keys())))
        state = ck.get("model")

        if isinstance(state, dict):
            info["model_tensor_count"] = len(state)
            info["model_parameter_values"] = int(
                sum(v.numel() for v in state.values() if torch.is_tensor(v))
            )
            info["first_state_keys"] = json.dumps(list(state.keys())[:12])
        else:
            info["model_tensor_count"] = np.nan
            info["model_parameter_values"] = np.nan
            info["first_state_keys"] = None

        info["has_fingerprint"] = "fingerprint" in ck
        info["has_optimizer"] = any(
            k in ck for k in ("optimizer", "optimizer_state_dict", "optim")
        )
        info["has_scheduler"] = any(
            k in ck for k in ("scheduler", "scheduler_state_dict")
        )
        info["has_epoch"] = "epoch" in ck
    else:
        info["keys"] = None
        info["model_tensor_count"] = np.nan
        info["model_parameter_values"] = np.nan
        info["first_state_keys"] = None
        info["has_fingerprint"] = False
        info["has_optimizer"] = False
        info["has_scheduler"] = False
        info["has_epoch"] = False

    del ck
    gc.collect()
    return info


# One representative checkpoint per (fold, pixel_group) is enough for structure audit.
audit_indices = (
    registry.sort_values(["fold", "holdout"], ascending=[True, False])
    .drop_duplicates(subset=["fold", "pixel_group"])
    ["member_index"]
    .astype(int)
    .tolist()
)

checkpoint_audit = []
for idx in audit_indices:
    m = members[idx]
    p = WEIGHTS / str(m["file"])
    print("auditing:", p.name)
    row = summarize_checkpoint(p)
    row["member_id"] = m.get("id")
    row["fold"] = m.get("fold")
    checkpoint_audit.append(row)

checkpoint_audit = pd.DataFrame(checkpoint_audit)
checkpoint_audit.to_csv(OUT / "checkpoint_structure_audit.csv", index=False)
display(checkpoint_audit)

if not checkpoint_audit.empty and not bool(checkpoint_audit["has_optimizer"].all()):
    print(
        "\nNOTE: at least one audited checkpoint has no saved optimizer state. "
        "That is fine: Notebook 02 can load model weights and create a NEW optimizer."
    )

## 5. Search for a real original study-to-fold map

A manifest member's `fold=2` tells us the checkpoint identity, **not which StudyInstanceUIDs belonged to fold 2**.

The code below only accepts a CSV as a fold map if it contains:

- `StudyInstanceUID`
- a column named `fold`, `Fold`, `fold_id`, or `cv_fold`

and each study maps to exactly one fold.

In [ ]:
FOLD_COLUMN_CANDIDATES = ["fold", "Fold", "fold_id", "cv_fold"]

def candidate_csvs(search_roots):
    seen = set()
    out = []

    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue

        # Avoid recursively traversing huge DICOM directories.
        for pattern in ("*.csv", "*/*.csv", "*/*/*.csv"):
            for p in root.glob(pattern):
                if any(part in {"train_series", "test_series"} for part in p.parts):
                    continue
                key = str(p.resolve())
                if key in seen:
                    continue
                seen.add(key)
                out.append(p)
    return out


def inspect_fold_csv(path: Path):
    try:
        x = pd.read_csv(path, nrows=20000, dtype={"StudyInstanceUID": str})
    except Exception:
        return None

    if "StudyInstanceUID" not in x.columns:
        return None

    fold_col = next((c for c in FOLD_COLUMN_CANDIDATES if c in x.columns), None)
    if fold_col is None:
        return None

    y = x[["StudyInstanceUID", fold_col]].dropna().copy()
    if y.empty:
        return None

    unique_per_uid = y.groupby("StudyInstanceUID")[fold_col].nunique()
    valid_unique = bool((unique_per_uid <= 1).all())

    return {
        "path": str(path),
        "fold_column": fold_col,
        "rows_read": len(x),
        "mapped_rows": len(y),
        "unique_uids": y["StudyInstanceUID"].nunique(),
        "one_fold_per_uid": valid_unique,
        "fold_values": json.dumps(sorted(map(str, y[fold_col].unique().tolist()))),
    }


fold_candidates = []
for p in candidate_csvs([WEIGHTS, ROOT]):
    info = inspect_fold_csv(p)
    if info is not None:
        fold_candidates.append(info)

fold_candidates_df = pd.DataFrame(fold_candidates)
fold_candidates_df.to_csv(OUT / "fold_artifact_candidates.csv", index=False)

if len(fold_candidates_df):
    display(fold_candidates_df)
else:
    print("No CSV study-to-fold artifact was found.")

accepted_fold_map = None
accepted_source = None

# Conservative automatic acceptance:
# require one-fold-per-UID and at least 90% coverage of the training UID set.
train_uids = set(train_df["StudyInstanceUID"].astype(str))

for info in fold_candidates:
    if not info["one_fold_per_uid"]:
        continue

    p = Path(info["path"])
    x = pd.read_csv(p, dtype={"StudyInstanceUID": str})
    fc = info["fold_column"]
    x = x[["StudyInstanceUID", fc]].dropna().drop_duplicates().copy()

    coverage = (
        len(train_uids.intersection(set(x["StudyInstanceUID"])))
        / max(len(train_uids), 1)
    )

    if coverage >= 0.90:
        x = x.rename(columns={fc: "fold"})
        accepted_fold_map = x
        accepted_source = str(p)
        break

HAS_SAFE_ORIGINAL_FOLD_MAP = accepted_fold_map is not None

if HAS_SAFE_ORIGINAL_FOLD_MAP:
    accepted_fold_map.to_csv(OUT / "original_fold_map.csv", index=False)
    print("Accepted fold map:", accepted_source)
    print("mapped studies:", accepted_fold_map["StudyInstanceUID"].nunique())
    print("fold counts:")
    display(accepted_fold_map.groupby("fold").size().rename("n_studies").to_frame())
else:
    print(
        "SAFE RESULT: no high-coverage original fold map was automatically verified.\n"
        "Notebook 02 must not invent a historical fold assignment."
    )

## 6. Build a resume manifest for Notebook 02

In [ ]:
resume_members = []
for m in members:
    cfg = m.get("config") or {}
    resume_members.append({
        "id": m.get("id"),
        "fold": m.get("fold"),
        "holdout": m.get("holdout"),
        "base_checkpoint_file": m.get("file"),
        "pixel_group": m.get("pixel_group"),
        "config": cfg,
        "finetuned_checkpoint_file": None,
        "finetuned_best_metric": None,
        "finetuned_epoch": None,
    })

resume_manifest = {
    "stage": "M0_BASELINE_AUDIT",
    "source_weight_package": str(WEIGHTS),
    "source_manifest": str(manifest_path),
    "seed": SEED,
    "targets": TARGETS,
    "n_train_studies": int(len(train_df)),
    "n_fully_annotated_12": int(len(expert_complete)),
    "has_verified_original_fold_map": bool(HAS_SAFE_ORIGINAL_FOLD_MAP),
    "verified_fold_map_source": accepted_source,
    "members": resume_members,
}

(OUT / "resume_manifest.json").write_text(json.dumps(resume_manifest, indent=2))
(OUT / "manifest_snapshot.json").write_text(json.dumps(manifest, indent=2))

print(json.dumps({
    "stage": resume_manifest["stage"],
    "members": len(resume_members),
    "n_train_studies": resume_manifest["n_train_studies"],
    "n_fully_annotated_12": resume_manifest["n_fully_annotated_12"],
    "has_verified_original_fold_map": resume_manifest["has_verified_original_fold_map"],
}, indent=2))

## 7. Write the Stage 1 safety contract

In [ ]:
status = {
    "stage1_ok": True,
    "competition_root": str(ROOT),
    "weights_root": str(WEIGHTS),
    "member_count": int(len(members)),
    "fold_ids_in_manifest": sorted(
        [
            x.item() if hasattr(x, "item") else x
            for x in registry["fold"].dropna().unique().tolist()
        ],
        key=lambda z: str(z),
    ),
    "targets": TARGETS,
    "masked_label_file": "train_labels_masked.csv",
    "expert_complete_file": "expert_complete_12.csv",
    "member_registry_file": "member_registry.csv",
    "resume_manifest_file": "resume_manifest.json",
    "original_fold_map_file": (
        "original_fold_map.csv" if HAS_SAFE_ORIGINAL_FOLD_MAP else None
    ),
    "has_verified_original_fold_map": bool(HAS_SAFE_ORIGINAL_FOLD_MAP),
    "next_stage_rule": (
        "Use verified historical fold mapping for fold-safe continuation."
        if HAS_SAFE_ORIGINAL_FOLD_MAP
        else
        "Do not claim fold-safe OOF from the base checkpoints until an original "
        "study-to-fold map or an explicitly independent validation design is supplied."
    ),
}

(OUT / "stage1_status.json").write_text(json.dumps(status, indent=2))

readme_lines = [
    "RSNA Knee continual fine-tuning - Stage 1 bundle",
    "",
    f"Competition root: {ROOT}",
    f"Weights root: {WEIGHTS}",
    f"Members: {len(members)}",
    f"Train studies: {len(train_df)}",
    f"Complete 12-target expert rows: {len(expert_complete)}",
    f"Verified original fold map: {HAS_SAFE_ORIGINAL_FOLD_MAP}",
    "",
    "Files:",
    "- member_registry.csv",
    "- checkpoint_structure_audit.csv",
    "- label_stats.csv",
    "- train_labels_masked.csv",
    "- expert_complete_12.csv",
    "- partial_or_incomplete_labels.csv",
    "- fold_artifact_candidates.csv",
    "- original_fold_map.csv (only when verified)",
    "- manifest_snapshot.json",
    "- resume_manifest.json",
    "- stage1_status.json",
    "",
    "Safety rule:",
    status["next_stage_rule"],
]
readme = "\n".join(readme_lines)
(OUT / "README.txt").write_text(readme)

print(readme)

## 8. Create the transfer bundle for Notebook 02

In [ ]:
bundle_path = WORK / "rsna_stage1_bundle.zip"
if bundle_path.exists():
    bundle_path.unlink()

shutil.make_archive(
    str(bundle_path.with_suffix("")),
    "zip",
    root_dir=OUT,
)

print("Created:", bundle_path)
print("Size MB:", round(bundle_path.stat().st_size / (1024**2), 3))

print("\nBundle contents:")
for p in sorted(OUT.iterdir()):
    print(" -", p.name)

## Notebook 01 output

The file to carry into the next stage is:

`/kaggle/working/rsna_stage1_bundle.zip`

The **original checkpoint files are not duplicated** into this ZIP because they are large and are already available through the existing `rsna-knee-weights` Kaggle input. Notebook 02 should mount both:

1. the original weight package; and
2. the Stage 1 bundle.

Notebook 02 will then implement **conservative fine-tuning**, after checking `stage1_status.json` and the validation/fold safety condition.